# Delta Lake MERGE Implementation

#Objective
Perform Incremental Data Processing using Delta Lake.

# Steps
1. Load Master Dataset
2. Clean the Dataset
3. Load Incremental Dataset
4. Perform MERGE Operation
5. Validate Results
6. Display Final Output

In [0]:
df_master=spark.read.csv("/Volumes/workspace/default/myfiles/customermaster.csv",header=True,inferSchema=True)
display(df_master)

customer_id,customer_name,city,age,email
101,Nethra,Hyderabad,21,nethra@gmail.com
102,Ashish,Chennai,22,ashish@gmail.com
103,Priya,Bangalore,22,priya@gmail.com
104,Dhruv,Delhi,30,dhruv@gmail.com
105,Anil,Mumbai,27,anil@gmail.com
105,Anil,Mumbai,27,anil@gmail.com
106,Sneha,null,24,sneha@gmail.com


In [0]:
df_master.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- email: string (nullable = true)



In [0]:
df_master.count()

7

In [0]:
from pyspark.sql.functions import *

df_master.select(
[
count(when(col(c).isNull(), c)).alias(c)
for c in df_master.columns
]
).show()

+-----------+-------------+----+---+-----+
|customer_id|customer_name|city|age|email|
+-----------+-------------+----+---+-----+
|          0|            0|   1|  0|    0|
+-----------+-------------+----+---+-----+



In [0]:
df_master=df_master.na.fill({
    "city":"unknown"
})
display(df_master)

customer_id,customer_name,city,age,email
101,Nethra,Hyderabad,21,nethra@gmail.com
102,Ashish,Chennai,22,ashish@gmail.com
103,Priya,Bangalore,22,priya@gmail.com
104,Dhruv,Delhi,30,dhruv@gmail.com
105,Anil,Mumbai,27,anil@gmail.com
105,Anil,Mumbai,27,anil@gmail.com
106,Sneha,unknown,24,sneha@gmail.com


In [0]:
df_master=df_master.dropDuplicates()
display(df_master)

customer_id,customer_name,city,age,email
105,Anil,Mumbai,27,anil@gmail.com
103,Priya,Bangalore,22,priya@gmail.com
104,Dhruv,Delhi,30,dhruv@gmail.com
101,Nethra,Hyderabad,21,nethra@gmail.com
102,Ashish,Chennai,22,ashish@gmail.com
106,Sneha,unknown,24,sneha@gmail.com


In [0]:
df_master.write.format("delta").mode("overwrite").save("/Volumes/workspace/default/myfiles/customer_delta")


In [0]:
delta_df=spark.read.format("delta").load("/Volumes/workspace/default/myfiles/customer_delta")
display(delta_df)


customer_id,customer_name,city,age,email
105,Anil,Mumbai,27,anil@gmail.com
103,Priya,Bangalore,22,priya@gmail.com
104,Dhruv,Delhi,30,dhruv@gmail.com
101,Nethra,Hyderabad,21,nethra@gmail.com
102,Ashish,Chennai,22,ashish@gmail.com
106,Sneha,unknown,24,sneha@gmail.com


In [0]:
incremental_df=spark.read.csv("/Volumes/workspace/default/myfiles/customerincremental.csv",header=True,inferSchema=True)
display(incremental_df)

customer_id,customer_name,city,age,email
102,Ashish,Hyderabad,25,ashish@gmail.com
104,Dhruv,Pune,31,kiran@gmail.com
107,Arjun,Vizag,24,arjun@gmail.com
108,Meera,Kochi,23,meera@gmail.com


In [0]:
from delta.tables import DeltaTable
deltatable=DeltaTable.forPath(spark,"/Volumes/workspace/default/myfiles/customer_delta")

In [0]:
deltatable.alias("target").merge(incremental_df.alias("source"),"target.customer_id=source.customer_id").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
final_df=spark.read.format("delta").load("/Volumes/workspace/default/myfiles/customer_delta")
display(final_df)

customer_id,customer_name,city,age,email
105,Anil,Mumbai,27,anil@gmail.com
103,Priya,Bangalore,22,priya@gmail.com
101,Nethra,Hyderabad,21,nethra@gmail.com
106,Sneha,unknown,24,sneha@gmail.com
102,Ashish,Hyderabad,25,ashish@gmail.com
104,Dhruv,Pune,31,kiran@gmail.com
107,Arjun,Vizag,24,arjun@gmail.com
108,Meera,Kochi,23,meera@gmail.com


In [0]:
print(final_df.count())

8


In [0]:
final_df.groupBy("customer_id").count().filter("count>1").show()

+-----------+-----+
|customer_id|count|
+-----------+-----+
+-----------+-----+



In [0]:
final_df.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- email: string (nullable = true)



In [0]:
display(final_df)

customer_id,customer_name,city,age,email
105,Anil,Mumbai,27,anil@gmail.com
103,Priya,Bangalore,22,priya@gmail.com
101,Nethra,Hyderabad,21,nethra@gmail.com
106,Sneha,unknown,24,sneha@gmail.com
102,Ashish,Hyderabad,25,ashish@gmail.com
104,Dhruv,Pune,31,kiran@gmail.com
107,Arjun,Vizag,24,arjun@gmail.com
108,Meera,Kochi,23,meera@gmail.com


Brief summary

The master customer dataset was loaded into a Delta table after cleaning null values and duplicate records. 
An incremental dataset containing updated and new customer records was then merged into the Delta table using the Delta Lake MERGE operation. Existing customer records were updated based on the customer ID, while new customer records were inserted. Finally, the output was validated by checking the row count, schema, duplicate records, and final dataset.